# Hacking Forums — 00: Extracción y Exploración

Caso 2 del curso de análisis forense de leaks underground.
Análisis comparativo de 7 foros de hacking con dumps SQL de distintas plataformas.

## Clasificación por tiers

Usamos el sistema de tiers del bloque transversal:

| Tier utilidad | Descripción |
|---|---|
| **A** | Dump SQL completo: usuarios + posts + threads + IPs. Permite análisis de red, atribución y estilometría. |
| **A↓** | Dump SQL parcial: solo usuarios. Sirve para pivoting de handles y análisis de persistencia, pero sin contenido. |
| **—** | Descartado: schema-only o muestra mínima sin valor analítico. |

| Tier técnica | Descripción |
|---|---|
| **B** | MyBB con prefijo no estándar. Requiere detección automática del prefijo de tablas. |
| **C** | IPS 3.x con variante de prefijo (`ibf_` o sin prefijo). Requiere ingeniería inversa del schema. |

---

### Datasets incluidos (Tier 1)

| Foro | Archivo | Parser | Utilidad | Técnica | Uso |
|---|---|---|---|---|---|
| **OGUsers** | OGUsers_2019.zip | MyBB | A | B | Red social, co-participación, pivoting |
| **Exploit.in** | Exploit.in_2013.12.13.zip | IPS 3.x ibf_ | A | C | Pivoting cross-foro (foro ruso) |
| **Cracked.to** | Cracked.to_2019.01.zip | MyBB | A | B | Pivoting cross-foro + contenido |
| **Nulled.io** | Nulled.io_2016.05.zip | IPS 3.x no-prefix | A | C | Pivoting cross-foro + contexto histórico |
| **RaidForums** | RaidForums_2021.zip | MyBB | A | B | Pivoting cross-foro |

### Datasets descartados

| Foro | Motivo |
|---|---|
| **Hackforums.net** (Hackforums.net_2015.zip) | Dump parcial (A↓): solo tabla `mybb_users`, sin posts ni threads. Sirve solo para persistencia de handles. |
| **Void.to** (Void.to.zip) | Schema-only: todas las tablas MyBB presentes pero cero `INSERT INTO`. Sin valor analítico. |

> **Nota**: Los foros con schema-only no contienen datos de usuarios ni de posts. Void.to se descarta completamente del análisis.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import networkx as nx

from src.utils import load_forum, list_forums, RESULTS_DIR, DATA_DIR

plt.style.use('dark_background')
sns.set_palette('muted')

HF_DIR = DATA_DIR / 'Hacking Forums'
HF_RESULTS = RESULTS_DIR / 'hacking_forums'
HF_RESULTS.mkdir(parents=True, exist_ok=True)

FORUMS_WITH_POSTS = [
    "OGUsers_2019.zip",
    "Exploit.in_2013.12.13.zip",
    "Cracked.to_2019.01.zip",
    "Nulled.io_2016.05.zip",
    "RaidForums_2021.zip",
]

print(f"DATA_DIR:    {DATA_DIR}")
print(f"HF_DIR:      {HF_DIR}")
print(f"HF_RESULTS:  {HF_RESULTS}")

## Sección 1 — Carga de datos

El parser auto-detecta el formato del dump SQL (MyBB, IPS 3.x con o sin prefijo `ibf_`).

In [ ]:
all_paths = list_forums("Hacking Forums")
selected_paths = [p for p in all_paths if p.name in FORUMS_WITH_POSTS]
print(f"Archivos seleccionados: {len(selected_paths)}")
for p in sorted(selected_paths, key=lambda x: x.name):
    print(f"  {p.name}")

print()
forums_data = {}
for path in selected_paths:
    stem = path.stem
    try:
        dfs = load_forum(path)
        if not dfs:
            print(f"  ⚠  {stem}: schema-only (sin datos)")
            continue
        forums_data[stem] = dfs
        u = len(dfs.get('user', []))
        p = len(dfs.get('post', []))
        t = len(dfs.get('thread', []))
        print(f"  ✓  {stem}: {u:,} usuarios  {p:,} posts  {t:,} threads")
    except Exception as e:
        print(f"  ✗  {stem}: {e}")

print(f"\nForos cargados: {len(forums_data)}")

## Sección 2 — EDA por foro

Estadística descriptiva comparada: usuarios, posts, threads y longitud de posts.

In [ ]:
rows = []
for name, dfs in forums_data.items():
    user_df  = dfs.get('user',   pd.DataFrame())
    post_df  = dfs.get('post',   pd.DataFrame())
    thread_df = dfs.get('thread', pd.DataFrame())

    n_users   = len(user_df)
    n_posts   = len(post_df)
    n_threads = len(thread_df)

    # Date range from posts
    date_start = date_end = None
    text_col = None
    if not post_df.empty:
        for col in ('dateline', 'post_date', 'date'):
            if col in post_df.columns:
                ts = pd.to_datetime(post_df[col], errors='coerce', utc=True)
                valid = ts.dropna()
                if len(valid):
                    date_start = valid.min().strftime('%Y-%m')
                    date_end   = valid.max().strftime('%Y-%m')
                break
        for col in ('pagetext', 'message', 'post_content'):
            if col in post_df.columns:
                text_col = col
                break

    avg_len = None
    if text_col and not post_df.empty:
        sample = post_df[text_col].dropna().astype(str)
        if len(sample):
            avg_len = round(sample.str.len().mean(), 0)

    rows.append({
        'Forum':          name,
        'Users':          n_users,
        'Posts':          n_posts,
        'Threads':        n_threads,
        'First post':     date_start or '—',
        'Last post':      date_end   or '—',
        'Avg post length': avg_len   or '—',
    })

summary_df = pd.DataFrame(rows).set_index('Forum')
print(summary_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

forum_names = list(forums_data.keys())
post_counts = [len(forums_data[f].get('post',   pd.DataFrame())) for f in forum_names]
user_counts = [len(forums_data[f].get('user',   pd.DataFrame())) for f in forum_names]
short_names = [f.split('_')[0] for f in forum_names]

bars0 = axes[0].bar(short_names, post_counts, color='#4E9EE9', alpha=0.85)
axes[0].bar_label(bars0, fmt='{:,.0f}', padding=3, fontsize=8)
axes[0].set_title('Posts por foro')
axes[0].set_ylabel('Posts')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1_000_000:.1f}M' if v >= 1e6 else f'{v/1000:.0f}K'))
axes[0].tick_params(axis='x', rotation=30)

bars1 = axes[1].bar(short_names, user_counts, color='#E9A24E', alpha=0.85)
axes[1].bar_label(bars1, fmt='{:,.0f}', padding=3, fontsize=8)
axes[1].set_title('Usuarios por foro')
axes[1].set_ylabel('Usuarios')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1000:.0f}K'))
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Comparativa de tamaño entre foros', y=1.01)
plt.tight_layout()
plt.savefig(HF_RESULTS / 'hf_size_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

colors = ['#4E9EE9', '#E94E4E', '#4EE97A', '#E9A24E', '#A24EE9']
any_plotted = False

for (name, dfs), color in zip(forums_data.items(), colors):
    post_df = dfs.get('post', pd.DataFrame())
    if post_df.empty:
        continue
    for col in ('dateline', 'post_date', 'date'):
        if col in post_df.columns:
            ts = pd.to_datetime(post_df[col], errors='coerce', utc=True)
            valid = post_df.loc[ts.notna()].copy()
            valid['month'] = ts[ts.notna()].dt.to_period('M')
            monthly = valid.groupby('month').size()
            monthly.index = monthly.index.to_timestamp()
            short = name.split('_')[0]
            ax.plot(monthly.index, monthly.values, label=short, color=color,
                    linewidth=1.4, alpha=0.85)
            any_plotted = True
            break

if any_plotted:
    ax.set_title('Actividad mensual por foro (posts/mes)')
    ax.set_xlabel('Fecha')
    ax.set_ylabel('Posts / mes')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1000:.0f}K'))
    ax.legend()
    plt.tight_layout()
    plt.savefig(HF_RESULTS / 'hf_temporal_activity.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Sin datos de fecha disponibles.")

## Sección 3 — Red de co-participación

Construimos el grafo de **co-participación en threads**: dos usuarios están conectados
si postearon en el mismo hilo. Usamos OGUsers (el foro más grande y con más posts)
para mantener el análisis manejable en memoria.

Los **top 150 nodos por degree** se visualizan de forma interactiva con Plotly.
El color y tamaño de cada nodo indica su degree (número de conexiones).

In [ ]:
# Identify the OGUsers key (has 'OGUsers' in stem and has posts)
og_key = next(
    (k for k in forums_data if 'oguser' in k.lower() and 'post' in forums_data[k]),
    None
)

if og_key is None:
    og_key = next((k for k in forums_data if 'post' in forums_data[k]), None)

G = nx.Graph()
uid_to_name = {}
cent_df = pd.DataFrame()

if og_key:
    posts = forums_data[og_key]['post']
    user_df = forums_data[og_key].get('user', pd.DataFrame())
    if not user_df.empty and 'userid' in user_df.columns and 'username' in user_df.columns:
        uid_to_name = user_df.set_index('userid')['username'].to_dict()

    thread_users = posts.groupby('threadid')['userid'].apply(list)
    thread_users = thread_users[thread_users.apply(lambda x: len(set(x)) >= 2)]
    print(f"Threads con ≥2 participantes: {len(thread_users):,}")

    # Sample up to 5000 threads for performance
    random.seed(42)
    sample_tids = list(thread_users.index)
    if len(sample_tids) > 5000:
        sample_tids = random.sample(sample_tids, 5000)

    for tid in sample_tids:
        parts = list(set(thread_users[tid]))
        for i in range(len(parts)):
            for j in range(i + 1, len(parts)):
                u, v = parts[i], parts[j]
                if G.has_edge(u, v):
                    G[u][v]['weight'] += 1
                else:
                    G.add_edge(u, v, weight=1)

    print(f"Grafo: {G.number_of_nodes():,} nodos, {G.number_of_edges():,} aristas")

    gcc = max(nx.connected_components(G), key=len)
    G_gcc = G.subgraph(gcc).copy()
    print(f"Componente gigante: {len(gcc):,} nodos ({len(gcc)/G.number_of_nodes()*100:.1f}%)")

    betw = nx.betweenness_centrality(G_gcc, k=min(200, len(gcc)), normalized=True, seed=42)

    cent_df = pd.DataFrame({
        'userid':       list(G_gcc.nodes()),
        'username':     [uid_to_name.get(n, str(n)) for n in G_gcc.nodes()],
        'degree':       [G_gcc.degree(n) for n in G_gcc.nodes()],
        'betweenness':  [betw[n] for n in G_gcc.nodes()],
    })

    print("\nTop 15 brokers (betweenness):")
    print(
        cent_df.nlargest(15, 'betweenness')[['username', 'betweenness', 'degree']]
        .to_string(index=False)
    )
else:
    print("No hay datos de posts disponibles para construir la red.")

In [ ]:
if G.number_of_nodes() > 0 and not cent_df.empty:
    import plotly.graph_objects as go

    top_nodes = cent_df.nlargest(150, 'degree')['userid'].tolist()
    G_sub = G.subgraph(top_nodes).copy()
    pos = nx.spring_layout(G_sub, seed=42, k=0.5)

    edge_x, edge_y = [], []
    for u, v in G_sub.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    node_x   = [pos[n][0] for n in G_sub.nodes()]
    node_y   = [pos[n][1] for n in G_sub.nodes()]
    node_deg = [G_sub.degree(n) for n in G_sub.nodes()]
    node_lbl = [uid_to_name.get(n, str(n)) for n in G_sub.nodes()]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=edge_x, y=edge_y, mode='lines',
        line=dict(color='#444', width=0.5), hoverinfo='none'
    ))
    fig.add_trace(go.Scatter(
        x=node_x, y=node_y, mode='markers+text',
        marker=dict(
            size=[max(4, d**0.5 * 2) for d in node_deg],
            color=node_deg, colorscale='YlOrRd', showscale=True,
            colorbar=dict(title='Degree'),
        ),
        text=node_lbl, textposition='top center', textfont=dict(size=7),
        hovertemplate='<b>%{text}</b><br>Degree: %{marker.color}<extra></extra>',
    ))
    fig.update_layout(
        title=f'Red de co-participación — {og_key or "foro"} (top 150 nodos)',
        template='plotly_dark', showlegend=False, width=950, height=650,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    )
    fig.show()
else:
    print("Sin grafo disponible — saltear visualización.")

## Sección 4 — Persistencia de handles

¿Los mismos usernames aparecen en múltiples foros?

Un handle que aparece en 2 o más foros distintos es una **señal de pivoting preliminar**.
La coincidencia exacta es conservadora pero de alta precisión; en secciones posteriores
se aplicará fuzzy matching para capturar variaciones.

In [ ]:
# Build per-forum handle sets (normalized: lowercase + strip)
handle_sets = {}
for name, dfs in forums_data.items():
    user_df = dfs.get('user', pd.DataFrame())
    if user_df.empty:
        continue
    for col in ('username', 'name', 'user_name'):
        if col in user_df.columns:
            handles = set(
                user_df[col].dropna().astype(str)
                .str.lower().str.strip()
                .tolist()
            )
            handles.discard('')
            handle_sets[name] = handles
            print(f"  {name}: {len(handles):,} handles únicos")
            break

# Cross-forum exact matches
forum_list = list(handle_sets.keys())
cross_rows = []

for i in range(len(forum_list)):
    for j in range(i + 1, len(forum_list)):
        fa, fb = forum_list[i], forum_list[j]
        shared = handle_sets[fa] & handle_sets[fb]
        cross_rows.append({
            'Forum A':  fa,
            'Forum B':  fb,
            'Shared handles': len(shared),
        })

cross_df = pd.DataFrame(cross_rows).sort_values('Shared handles', ascending=False)
print("\nHandles compartidos entre pares de foros:")
print(cross_df.to_string(index=False))

# Handles appearing in 3+ forums
from collections import Counter
all_handles = Counter()
for h_set in handle_sets.values():
    for h in h_set:
        all_handles[h] += 1

multi_forum = {h: c for h, c in all_handles.items() if c >= 2}
print(f"\nHandles en ≥2 foros: {len(multi_forum):,}")

# Top 30 by frequency across forums
top_multi = sorted(multi_forum.items(), key=lambda x: x[1], reverse=True)[:30]
top_df = pd.DataFrame(top_multi, columns=['Handle', 'Foros en los que aparece'])
print("\nTop 30 handles más ubicuos:")
print(top_df.to_string(index=False))

# Annotate which forums each top handle appears in
print("\nDetalle — handle → foros:")
for handle, _ in top_multi[:15]:
    forums_present = [f for f, h_set in handle_sets.items() if handle in h_set]
    print(f"  {handle!r:30s} → {', '.join(f.split('_')[0] for f in forums_present)}")

## Sección 5 — Detección de idioma por foro

**Paso obligatorio antes de cualquier análisis de contenido.** Burrows' Delta, NER y BERTopic son idioma-específicos: si un foro es mayoritariamente ruso, aplicarles el pipeline inglés produce resultados sin sentido.

Muestreamos hasta 500 posts por foro y aplicamos detección automática de idioma.

In [ ]:
import re, random
from langdetect import detect, LangDetectException

SAMPLE_PER_FORUM = 500
CHAR_SAMPLE = 2000

def strip_html(text):
    return re.sub(r"<[^>]+>", " ", str(text or ""))

lang_results = []

for name, dfs in forums_data.items():
    post_df = dfs.get('post', pd.DataFrame())
    if post_df.empty:
        print(f"[SKIP] {name}: sin posts")
        continue

    # Find text column
    text_col = next((c for c in ('pagetext', 'message', 'post_content') if c in post_df.columns), None)
    if text_col is None:
        print(f"[SKIP] {name}: no se encontró columna de texto")
        continue

    texts = post_df[text_col].dropna().astype(str).tolist()
    random.seed(42)
    sample = random.sample(texts, min(SAMPLE_PER_FORUM, len(texts)))

    lang_counts = {}
    for raw in sample:
        text = strip_html(raw)[:CHAR_SAMPLE].strip()
        if len(text) < 30:
            continue
        try:
            lang = detect(text)
        except LangDetectException:
            lang = "unknown"
        lang_counts[lang] = lang_counts.get(lang, 0) + 1

    total = sum(lang_counts.values()) or 1
    for lang, count in sorted(lang_counts.items(), key=lambda x: -x[1])[:5]:
        lang_results.append({
            "forum": name.split("_")[0],
            "lang": lang,
            "pct": round(count / total * 100, 1),
        })

lang_df = pd.DataFrame(lang_results)

if not lang_df.empty:
    pivot = lang_df.pivot_table(index="forum", columns="lang", values="pct", fill_value=0)

    pivot.plot(kind="bar", figsize=(12, 5), colormap="tab10")
    plt.title("Distribución de idioma por foro (muestra de posts)")
    plt.ylabel("% posts detectados")
    plt.xlabel("")
    plt.xticks(rotation=30, ha="right")
    plt.legend(title="idioma", bbox_to_anchor=(1.01, 1))
    plt.tight_layout()
    plt.savefig(HF_RESULTS / "hf_language_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\n=== Resumen de idioma por foro ===")
    for forum in pivot.index:
        top_lang = pivot.loc[forum].idxmax()
        top_pct = pivot.loc[forum].max()
        if top_lang == "ru":
            flag = "⚠️  RUSO — excluir de Delta/NER inglés; usar modelo multilingual"
        elif top_lang == "en":
            flag = "✓  inglés — pipeline estándar válido"
        else:
            flag = f"⚠️  {top_lang} — revisar pipeline"
        print(f"  {forum:<15} → {top_lang} ({top_pct:.0f}%)  {flag}")

    print("\n=== Decisión de pipeline ===")
    print("  Foros en inglés  → BERTopic (en), NER, Burrows' Delta")
    print("  Exploit.in (ru)  → qwen3-embedding multilingual; EXCLUIR de Delta y NER inglés")
    print("  Embeddings       → qwen3-embedding (multilingual) para TODOS los foros")
else:
    print("No se pudo detectar idioma en ningún foro.")
